<!-- CELL-00 -->
# PRD-300 / CC-1.5.1 — NLP Backfill pe GPU

Rulează pipeline-ul rhetoric (FOMC-RoBERTa + Llama 3.3 70B) pe toate documentele FOMC (statements + minutes).

**Cerințe runtime:**
- Google Colab cu GPU (Runtime → Change runtime type → T4 GPU)
- Secrets setate în Colab (icon 🔑 sidebar stânga):
  - `HF_TOKEN` — token HuggingFace (hf_...)
  - `DEEPINFRA_API_KEY` — cheie API DeepInfra

**Timp estimat:** ~20-30 min (RoBERTa pe GPU: ~18 min, Llama API: ~15 min per statements, minutes din cache)

**Output:** `data/rhetoric/fomc_scores.parquet` — descarcă la final și pune în repo local.

In [ ]:
# CELL-01
print("[CELL-01]")

print("[CELL-01] Environment setup + GPU check")

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU available: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("NO GPU -- Runtime -> Change runtime type -> T4 GPU")
    print("  RoBERTa will run on CPU (~6 hours instead of ~18 minutes)")

# Verify Colab secrets
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    deepinfra_key = userdata.get("DEEPINFRA_API_KEY")

    if hf_token and hf_token.startswith("hf_"):
        print(f"HF_TOKEN: valid (hf_...{hf_token[-4:]})")
    else:
        print(f"HF_TOKEN: wrong prefix or missing -- needs hf_ prefix")

    if deepinfra_key and len(deepinfra_key) > 10:
        print(f"DEEPINFRA_API_KEY: present ({len(deepinfra_key)} chars)")
    else:
        print(f"DEEPINFRA_API_KEY: missing or short -- Llama scorer will be skipped")

    # Set as env vars for the pipeline
    import os
    os.environ["HF_TOKEN"] = hf_token or ""
    os.environ["DEEPINFRA_API_KEY"] = deepinfra_key if deepinfra_key else ""
    print("\nSecrets loaded into environment")

except Exception as e:
    print(f"Colab secrets error: {e}")
    print("  Go to sidebar key icon -> add HF_TOKEN and DEEPINFRA_API_KEY")

In [ ]:
# CELL-02
print("[CELL-02]")

print("[CELL-02] Clone repo + install")

import os

REPO_URL = "https://github.com/Inocenthhacker/macro_context_reader.git"
REPO_DIR = "/content/macro_context_reader"

if os.path.exists(REPO_DIR):
    print(f"Repo already cloned at {REPO_DIR}")
    os.chdir(REPO_DIR)
    !git pull --quiet
    print("Pulled latest changes")
else:
    !git clone {REPO_URL} {REPO_DIR} --quiet
    os.chdir(REPO_DIR)
    print(f"Cloned to {REPO_DIR}")

# Install project + dependencies
!pip install -e ".[dev]" --quiet 2>&1 | tail -3

# Verify key imports
from macro_context_reader.rhetoric.pipeline import run_full_pipeline
from macro_context_reader.rhetoric.scraper import fetch_fomc_statements, fetch_fomc_minutes
print("Pipeline imports OK")

# Show cached documents
from pathlib import Path
statements = list(Path("data/rhetoric/raw/statement").glob("*")) if Path("data/rhetoric/raw/statement").exists() else []
minutes = list(Path("data/rhetoric/raw/minutes").glob("*")) if Path("data/rhetoric/raw/minutes").exists() else []
print(f"Cached documents: {len(statements)} statements, {len(minutes)} minutes")

<!-- CELL-03 -->
## Pas 1: Verificare acces FOMC-RoBERTa

Descarcă modelul pe GPU. Prima rulare descarcă ~1.4 GB, ulterior e din cache.

In [ ]:
# CELL-04
print("[CELL-04]")

print("[CELL-03] Load FOMC-RoBERTa on GPU")

import os
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

token = os.environ.get("HF_TOKEN")
device = 0 if torch.cuda.is_available() else -1

print(f"Loading FOMC-RoBERTa (device={'GPU' if device == 0 else 'CPU'})...")
tokenizer = AutoTokenizer.from_pretrained("gtfintechlab/FOMC-RoBERTa", token=token)
model = AutoModelForSequenceClassification.from_pretrained("gtfintechlab/FOMC-RoBERTa", token=token)

if device == 0:
    model = model.cuda()

# Quick sanity test
inputs = tokenizer("Inflation remains elevated", return_tensors="pt", truncation=True)
if device == 0:
    inputs = {k: v.cuda() for k, v in inputs.items()}
with torch.no_grad():
    outputs = model(**inputs)
probs = torch.softmax(outputs.logits, dim=-1).squeeze().cpu().tolist()
print(f"Model loaded. Test: 'Inflation remains elevated' -> hawkish: {probs[1]:.3f}")

del model, tokenizer, inputs, outputs  # free memory for pipeline
torch.cuda.empty_cache()

<!-- CELL-05 -->
## Pas 2: Scrape documente lipsă (dacă e cazul)

Verifică că avem toate documentele cached. Dacă lipsesc, scrape-uiește.

In [ ]:
# CELL-06
print("[CELL-06]")

print("[CELL-04] Ensure all docs cached")

from macro_context_reader.rhetoric.scraper import fetch_fomc_statements, fetch_fomc_minutes
from pathlib import Path

# Check current cache
stmt_dir = Path("data/rhetoric/raw/statement")
mins_dir = Path("data/rhetoric/raw/minutes")

stmt_count = len(list(stmt_dir.glob("*"))) if stmt_dir.exists() else 0
mins_count = len(list(mins_dir.glob("*"))) if mins_dir.exists() else 0

print(f"Current cache: {stmt_count} statements, {mins_count} minutes")

if stmt_count < 40:
    print("Fetching statements...")
    stmts = fetch_fomc_statements(start_year=2021)
    print(f"  Fetched {len(stmts)} statements")
else:
    print("Statements cache OK")

if mins_count < 40:
    print("Fetching minutes...")
    mins = fetch_fomc_minutes(start_year=2021)
    print(f"  Fetched {len(mins)} minutes")
else:
    print("Minutes cache OK")

# Final count
stmt_count = len(list(stmt_dir.glob("*"))) if stmt_dir.exists() else 0
mins_count = len(list(mins_dir.glob("*"))) if mins_dir.exists() else 0
print(f"\nTotal cached: {stmt_count} statements + {mins_count} minutes = {stmt_count + mins_count} documents")

<!-- CELL-07 -->
## Pas 3: Rulează pipeline-ul complet

FOMC-RoBERTa (GPU) + Llama 3.3 70B (DeepInfra API).

**Timp estimat:** ~20-30 min.

Dacă Llama eșuează (balance insuficient sau rate limit), pipeline-ul continuă cu RoBERTa-only.

In [ ]:
# CELL-08
print("[CELL-08]")

print("[CELL-05] Run full NLP pipeline")

import logging
import time
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)

from macro_context_reader.rhetoric.pipeline import run_full_pipeline

output_path = Path("data/rhetoric/fomc_scores.parquet")

start = time.time()

try:
    result = run_full_pipeline(
        start_year=2021,
        doc_types=["statement", "minutes"],
        force_refetch=False,
        output_path=output_path,
    )
    elapsed = time.time() - start

    print(f"\n{'='*60}")
    print(f"PIPELINE COMPLETE -- {elapsed/60:.1f} minutes")
    print(f"{'='*60}")
    print(f"Documents scored: {len(result)}")
    print(f"Columns: {result.columns.tolist()}")
    if len(result) > 0:
        print(f"Date range: {result['date'].min()} to {result['date'].max()}")
        if 'doc_type' in result.columns:
            print(f"\nBy doc_type:")
            print(result['doc_type'].value_counts().to_string())

        score_cols = [c for c in result.columns if 'net' in c.lower()]
        for col in score_cols:
            print(f"\n{col}: min={result[col].min():.3f}, max={result[col].max():.3f}, mean={result[col].mean():.3f}")

except Exception as e:
    elapsed = time.time() - start
    print(f"\nPipeline FAILED after {elapsed/60:.1f} minutes")
    print(f"Error: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

    # Attempt RoBERTa-only fallback
    print(f"\nAttempting RoBERTa-only fallback...")
    try:
        result = run_full_pipeline(
            start_year=2021,
            doc_types=["statement", "minutes"],
            scorer_names=["fomc_roberta"],
            force_refetch=False,
            output_path=output_path,
        )
        print(f"Fallback succeeded: {len(result)} documents scored with RoBERTa-only")
    except Exception as e2:
        print(f"Fallback also failed: {e2}")
        raise

<!-- CELL-09 -->
## Pas 4: Validare + Spot checks

In [ ]:
# CELL-10
print("[CELL-10]")

print("[CELL-06] Validation + spot checks")

import pandas as pd

df = pd.read_parquet("data/rhetoric/fomc_scores.parquet")

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
null_counts = df.isna().sum()
if null_counts.any():
    print(f"Null counts:\n{null_counts[null_counts > 0]}")
else:
    print("No nulls")

# Spot checks
checks = {
    "2022-03-16": "hawkish (hiking cycle start)",
    "2024-09-18": "dovish (cutting cycle start)",
}

for date_str, expected in checks.items():
    ts = pd.Timestamp(date_str)
    match = df[df['date'] == ts]
    if len(match) > 0:
        row = match.iloc[0]
        print(f"\n{'='*50}")
        print(f"Spot check: {date_str} -- expected {expected}")
        for col in ['ensemble_net', 'weighted_score', 'confidence']:
            if col in row.index:
                print(f"  {col}: {row[col]}")
        net_cols = [c for c in row.index if '_net' in c]
        for col in net_cols:
            print(f"  {col}: {row[col]:.3f}")
    else:
        print(f"\n{date_str} not found in data")

# Coverage check
if 'doc_type' in df.columns:
    stmt_count = (df['doc_type'] == 'statement').sum()
    mins_count = (df['doc_type'] == 'minutes').sum()
    print(f"\nCoverage: {stmt_count} statements + {mins_count} minutes = {len(df)} total")
    if len(df) >= 60:
        print(f"Meets >=60 threshold")
    else:
        print(f"Below >=60 threshold (got {len(df)})")

<!-- CELL-11 -->
## Pas 5: Descarcă parquet

Descarcă `fomc_scores.parquet` și pune-l în `data/rhetoric/` din repo-ul local.
Apoi rulează local:
```bash
git add -f data/rhetoric/fomc_scores.parquet
git commit -m "feat(rhetoric): NLP backfill -- 84+ docs scored on Colab T4 GPU [PRD-300/CC-1.5.1]"
git push origin main
```

In [ ]:
# CELL-12
print("[CELL-12]")

print("[CELL-07] Download parquet")

from pathlib import Path

parquet_path = Path("data/rhetoric/fomc_scores.parquet")

if parquet_path.exists():
    size_kb = parquet_path.stat().st_size / 1024
    print(f"File: {parquet_path}")
    print(f"Size: {size_kb:.1f} KB")

    try:
        from google.colab import files
        files.download(str(parquet_path))
        print("Download triggered -- check your browser downloads")
    except ImportError:
        print("Not running in Colab -- file available at:")
        print(f"  {parquet_path.resolve()}")
else:
    print("Parquet not found -- pipeline may have failed")